# Example 0.13.2 / 5: Exercise 8-1: Starting from the Middle of the Bounds

Companion notebook for **0.13.2 Exercises on Sets, Parameters and Variables** in the AMPL lab Documentation, section "Exercise 8-1: Starting from the Middle of the Bounds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 8-1: Starting from the Middle of the Bounds

Exercise 8-1 initializes each variable of the diet model with unions
(Set Operations (AMPL lab documentation, section 4.1)) midway between its bounds, then asks which constraints that starting
point violates and how its cost compares with the optimum. The `:=` phrase in a `var` declaration
sets the initial value (Defined Variables and Initial Values (AMPL lab documentation, section 1.3.2)). The data are those of
the book's Figure 5-2.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MINREQ;                       # nutrients with a minimum requirement
set MAXREQ;                       # nutrients with a maximum allowance
set NUTR = MINREQ union MAXREQ;   # all nutrients
set FOOD;                         # foods
param cost {FOOD} > 0;            # dollars per package
param f_min {FOOD} >= 0;          # fewest packages
param f_max {j in FOOD} >= f_min[j];  # most packages
param n_min {MINREQ} >= 0;        # minimum amount of a nutrient
param n_max {MAXREQ} >= 0;        # maximum amount of a nutrient
param amt {NUTR, FOOD} >= 0;      # nutrient per package

# packages bought, starting midway between the bounds
var Buy {j in FOOD} >= f_min[j], <= f_max[j], := (f_min[j] + f_max[j]) / 2;

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];  # dollars

subject to Diet_Min {i in MINREQ}: sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];  # minimums
subject to Diet_Max {i in MAXREQ}: sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];  # maximums

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

foods = pd.DataFrame(
    {"cost": [3.19, 2.59, 2.29, 2.89, 1.89, 1.99, 1.99, 2.49], "f_min": 2, "f_max": 10},
    index=["BEEF", "CHK", "FISH", "HAM", "MCH", "MTL", "SPG", "TUR"],
)
amt = pd.DataFrame(
    {"BEEF": [60, 20, 10, 15, 938, 295], "CHK": [8, 0, 20, 20, 2180, 770],
     "FISH": [8, 10, 15, 10, 945, 440], "HAM": [40, 40, 35, 10, 278, 430],
     "MCH": [15, 35, 15, 15, 1182, 315], "MTL": [70, 30, 15, 15, 896, 400],
     "SPG": [25, 50, 25, 15, 1329, 370], "TUR": [60, 20, 15, 10, 1397, 450]},
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)
ampl.set["MINREQ"] = ["A", "B1", "B2", "C", "CAL"]
ampl.set["MAXREQ"] = ["A", "NA", "CAL"]
ampl.set["FOOD"] = list(foods.index)
ampl.param["cost"] = foods["cost"]
ampl.param["f_min"] = foods["f_min"]
ampl.param["f_max"] = foods["f_max"]
ampl.param["n_min"] = {"A": 700, "C": 700, "B1": 0, "B2": 0, "CAL": 16000}
ampl.param["n_max"] = {"A": 20000, "NA": 50000, "CAL": 24000}
ampl.param["amt"] = amt

In [5]:
# run.py
print("start cost:", round(ampl.obj["Total_Cost"].value(), 2))
print("min slacks:", ampl.con["Diet_Min"].get_values(["slack"]).to_dict())
print("max slacks:", ampl.con["Diet_Max"].get_values(["slack"]).to_dict())
ampl.solve(solver="highs")
print(ampl.solve_result, "optimal cost:", round(ampl.obj["Total_Cost"].value(), 2))

start cost: 115.92
min slacks: {'A': 1016, 'B1': 900, 'B2': 660, 'C': 530, 'CAL': 4820}
max slacks: {'A': 18284, 'CAL': 3180, 'NA': -4870}
HiGHS 1.14.0: optimal solution; objective 74.27382022
2 simplex iterations
0 barrier iterations
solved optimal cost: 74.27


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set MINREQ := A B1 B2 C CAL;
set MAXREQ := A NA CAL;
set FOOD := BEEF CHK FISH HAM MCH MTL SPG TUR;
param:  cost  f_min  f_max :=
BEEF    3.19    2     10
CHK     2.59    2     10
FISH    2.29    2     10
HAM     2.89    2     10
MCH     1.89    2     10
MTL     1.99    2     10
SPG     1.99    2     10
TUR     2.49    2     10;
param:  n_min  n_max :=
A         700  20000
C         700      .
B1          0      .
B2          0      .
NA          .  50000
CAL     16000  24000;
param amt (tr):  A   C   B1  B2    NA   CAL :=
BEEF            60  20  10  15   938   295
CHK              8   0  20  20  2180   770
FISH             8  10  15  10   945   440
HAM             40  40  35  10   278   430
MCH             15  35  15  15  1182   315
MTL             70  30  15  15   896   400
SPG             25  50  25  15  1329   370
TUR             60  20  15  10  1397   450;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display Total_Cost;
display Diet_Min.slack, Diet_Max.slack;
option solver highs;
solve;
display Total_Cost;

Total_Cost = 115.92

:   Diet_Min.slack Diet_Max.slack    :=
A         1016          18284
B1         900             .
B2         660             .
C          530             .
CAL       4820           3180
NA          .           -4870
;

HiGHS 1.14.0: optimal solution; objective 74.27382022
2 simplex iterations
0 barrier iterations
Total_Cost = 74.2738

